# Capstone One — Latent Neural ODE for Malware Concept Drift

Single Colab notebook reproducing the full experiment:
- Chronological BODMAS split (train 2019-08..2020-03, test Jun–Sep 2020)
- Static baselines: XGBoost, LightGBM, parameter-matched MLP
- Latent Neural ODE with adjoint sensitivity (dopri5 / rk4) + Focal Loss
- Ablation: decoupled encoder/decoder (no ODE)
- Month-by-month AUROC / AUT plots and UMAP latent-space migration

All metrics are computed from real evaluated results (no synthetic data).

In [ ]:
# Install required packages (uncomment if running on a fresh Colab runtime)
!pip -q install torch torchdiffeq umap-learn xgboost lightgbm scikit-learn pandas matplotlib
!nvidia-smi 2>/dev/null || echo "No GPU detected — running on CPU (small configs used)"

In [ ]:
import os, glob, json, numpy as np, pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torchdiffeq import odeint_adjoint
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
try:
    import xgboost as xgb
    HAS_XGB = True
except Exception:
    HAS_XGB = False
try:
    import lightgbm as lgb
    HAS_LGBM = True
except Exception:
    HAS_LGBM = False
try:
    import umap
    HAS_UMAP = True
except Exception:
    HAS_UMAP = False

DEVICE = 'cpu'
RANDOM = 42
np.random.seed(RANDOM); torch.manual_seed(RANDOM)

os.makedirs('E:\capstone/results', exist_ok=True)
os.makedirs('E:\capstone/graphs', exist_ok=True)
print('DEVICE:', DEVICE, '| xgboost:', HAS_XGB, '| lightgbm:', HAS_LGBM, '| umap:', HAS_UMAP)

In [ ]:
BASE_DIR = 'E:\capstone'
PARQUET_PATH = os.path.join(BASE_DIR, 'dataset', 'bodmas_clean_features.parquet')
RANKING_PATH = os.path.join(BASE_DIR, 'dataset', 'feature_drift_ranking.csv')

TRAIN_END = pd.Period('2020-03', 'M')
TEST_START = pd.Period('2020-06', 'M')

def load_dataframe(max_features=600):
    df = pd.read_parquet(PARQUET_PATH)
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df['month'] = df['timestamp'].dt.to_period('M')
    import pyarrow.parquet as pq
    available = set(pq.ParquetFile(PARQUET_PATH).schema_arrow.names)
    top = None
    if os.path.exists(RANKING_PATH) and max_features:
        rank = pd.read_csv(RANKING_PATH)
        names = rank.iloc[:, 0].astype(str).tolist()
        top = [c for c in names[:max_features] if c in available]
    meta = ['sha256', 'Label', 'timestamp']
    cols = [c for c in meta + (top or []) if c in available]
    if cols:
        df = df[cols]
    feat_cols = [c for c in df.columns if c.startswith('F') and c[1:].isdigit()]
    return df, feat_cols

def split_chronological(df):
    train_mask = df['month'] <= TRAIN_END
    val_mask = (df['month'] >= pd.Period('2020-04','M')) & (df['month'] <= pd.Period('2020-05','M'))
    test_mask = df['month'] >= TEST_START
    test_months = sorted(df.loc[test_mask,'month'].unique().astype(str))
    month_idx = {m: df.index[test_mask & (df['month'].astype(str)==m)].tolist() for m in test_months}
    return {'train': train_mask, 'val': val_mask, 'test': test_mask,
            'test_months': test_months, 'month_idx': month_idx}

def get_arrays(df, feat_cols, mask):
    X = df.loc[mask, feat_cols].fillna(0).astype(np.float32).values
    y = df.loc[mask, 'Label'].astype(np.float32).values
    t = df.loc[mask, 'timestamp'].values
    return X, y, t

def normalize_time(ts):
    ts = pd.to_datetime(ts)
    t0, t1 = ts.min(), ts.max()
    span = (t1 - t0).total_seconds()
    if span == 0:
        return np.zeros(len(ts), dtype=np.float32)
    return np.asarray((ts - t0).total_seconds() / span, dtype=np.float32)

df, feat_cols = load_dataframe(max_features=600)
split_info = split_chronological(df)
print('Features:', len(feat_cols), '| Train:', split_info['train'].sum(),
      '| Val:', split_info['val'].sum(), '| Test:', split_info['test'].sum())
print('Test months:', split_info['test_months'])

In [ ]:
class ODEFunc(nn.Module):
    def __init__(self, latent_dim, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, hidden), nn.Tanh(),
            nn.Linear(hidden, hidden), nn.Tanh(),
            nn.Linear(hidden, latent_dim),
        )
        nn.init.zeros_(self.net[-1].weight); nn.init.zeros_(self.net[-1].bias)
    def forward(self, t, z):
        return self.net(z)

class LatentNeuralODE(nn.Module):
    def __init__(self, input_dim, latent_dim=8, hidden=32, solver='dopri5',
                 use_ode=True, atol=1e-3, rtol=1e-3):
        super().__init__()
        self.solver = solver; self.use_ode = use_ode
        self.atol = atol; self.rtol = rtol
        decoder_in = latent_dim + (1 if use_ode else 0)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, latent_dim),
        )
        self.ode_func = ODEFunc(latent_dim, hidden=hidden)
        self.decoder = nn.Sequential(
            nn.Linear(decoder_in, hidden), nn.ReLU(), nn.Linear(hidden, 1),
        )
    def forward(self, x, t):
        z0 = self.encoder(x)
        if self.use_ode:
            T = torch.tensor([0.0, 1.0], device=x.device, dtype=x.dtype)
            z_traj = odeint_adjoint(self.ode_func, z0, T,
                                    method=self.solver, atol=self.atol, rtol=self.rtol)
            z_t = z_traj[-1]
            z_in = torch.cat([z_t, t.unsqueeze(-1)], dim=-1)
        else:
            z_in = z0
        logits = self.decoder(z_in).squeeze(-1)
        return logits, z0, z_t
    def latent_states(self, x, t):
        with torch.no_grad():
            _, z0, zt = self.forward(x, t)
        return z0, zt

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma
    def forward(self, logits, targets):
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        pt = torch.exp(-bce)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()

print('ODE model classes defined')

In [ ]:
def safe_auroc(y, p):
    if len(np.unique(y)) < 2: return float('nan')
    return float(roc_auc_score(y, p))

def safe_f1(y, p):
    pred = (p >= 0.5).astype(int)
    if len(np.unique(y)) < 2: return float('nan')
    return float(f1_score(y, pred, average='macro', zero_division=0))

def evaluate_model(predict_fn, df, feat_cols, split_info):
    res = {'months': [], 'per_month': {}}
    all_y, all_p = [], []
    for m in split_info['test_months']:
        idx = split_info['month_idx'][m]
        X = df.loc[idx, feat_cols].fillna(0).astype(np.float32).values
        y = df.loc[idx, 'Label'].astype(np.float32).values
        t = normalize_time(df.loc[idx, 'timestamp'].values)
        p = np.asarray(predict_fn(X, t), dtype=np.float64).ravel()
        res['per_month'][m] = {
            'auroc': safe_auroc(y, p), 'f1_macro': safe_f1(y, p),
            'support': int(len(y)), 'malware': int((y==1).sum()), 'benign': int((y==0).sum()),
        }
        res['months'].append(m)
        all_y.append(y); all_p.append(p)
    aurocs = [res['per_month'][m]['auroc'] for m in split_info['test_months']]
    valid = [a for a in aurocs if not np.isnan(a)]
    aut = float(np.trapz(valid, x=np.arange(len(valid)))) if len(valid) >= 2 else (
        float(valid[0]) if len(valid) == 1 else float('nan'))
    all_y = np.concatenate(all_y); all_p = np.concatenate(all_p)
    res['overall'] = {
        'auroc': safe_auroc(all_y, all_p), 'f1_macro': safe_f1(all_y, all_p),
        'aut': aut,
        'auroc_per_month': [res['per_month'][m]['auroc'] for m in split_info['test_months']],
    }
    return res

print('Evaluation utilities ready')

In [ ]:
def subsample(X, y, t, max_samples):
    n = X.shape[0]
    if n <= max_samples: return X, y, t
    rng = np.random.RandomState(RANDOM)
    idx = rng.choice(n, size=max_samples, replace=False)
    return X[idx], y[idx], t[idx]

def train_and_eval_baselines(epochs_mlp=300):
    Xtr, ytr, _ = get_arrays(df, feat_cols, split_info['train'])
    scaler = StandardScaler()
    Xtr_s = scaler.fit_transform(Xtr)
    models = {}
    if HAS_XGB:
        models['XGBoost (static)'] = xgb.XGBClassifier(
            n_estimators=300, max_depth=6, learning_rate=0.1, subsample=0.8,
            colsample_bytree=0.8, random_state=RANDOM, n_jobs=-1, verbosity=0)
    if HAS_LGBM:
        models['LightGBM (static)'] = lgb.LGBMClassifier(
            n_estimators=300, num_leaves=64, learning_rate=0.1, subsample=0.8,
            colsample_bytree=0.8, random_state=RANDOM, n_jobs=-1, verbosity=-1)
    models['MLP (static)'] = MLPClassifier(
        hidden_layer_sizes=(128,64), max_iter=epochs_mlp, random_state=RANDOM,
        early_stopping=True)
    rows = []
    for name, model in models.items():
        Xfit = Xtr_s if 'MLP' in name else Xtr
        model.fit(Xfit, ytr)
        use_scaler = 'MLP' in name
        def make_pf(m, us):
            def pf(X, t):
                Xf = scaler.transform(X) if us else X
                return m.predict_proba(Xf)[:, 1]
            return pf
        res = evaluate_model(make_pf(model, use_scaler), df, feat_cols, split_info)
        ov = res['overall']
        print(f"{name}: AUROC={ov['auroc']:.4f} F1={ov['f1_macro']:.4f} AUT={ov['aut']:.4f}")
        for m in res['months']:
            pm = res['per_month'][m]
            rows.append({'model': name, 'month': m, 'auroc': pm['auroc'],
                         'f1_macro': pm['f1_macro'], 'support': pm['support'],
                         'malware': pm['malware'], 'benign': pm['benign']})
        rows.append({'model': name, 'month': 'OVERALL', 'auroc': ov['auroc'],
                     'f1_macro': ov['f1_macro'], 'aut': ov['aut'],
                     'support': res['per_month'][split_info['test_months'][0]]['support'],
                     'malware': sum(r['malware'] for r in rows if r['model']==name and r['month']!='OVERALL'),
                     'benign': sum(r['benign'] for r in rows if r['model']==name and r['month']!='OVERALL')})
    pd.DataFrame(rows).to_csv('E:\capstone/results/baselines_metrics.csv', index=False)
    return models, scaler

baseline_models, scaler = train_and_eval_baselines()

In [ ]:
def train_epoch(model, loader, opt, crit):
    model.train(); tot, nb = 0.0, 0
    for Xb, yb, tb in loader:
        Xb, yb, tb = Xb.to(DEVICE), yb.to(DEVICE), tb.to(DEVICE)
        opt.zero_grad()
        logits, _, _ = model(Xb, tb)
        loss = crit(logits, yb)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        tot += float(loss.detach()); nb += 1
    return tot / max(nb, 1)

def val_loss(model, Xv, yv, tv, crit, bs=1024):
    model.eval(); tot, n = 0.0, 0
    with torch.no_grad():
        for i in range(0, Xv.shape[0], bs):
            Xb = torch.from_numpy(Xv[i:i+bs]).to(DEVICE)
            yb = torch.from_numpy(yv[i:i+bs]).to(DEVICE)
            tb = torch.from_numpy(tv[i:i+bs]).to(DEVICE)
            lg, _, _ = model(Xb, tb)
            tot += float(crit(lg, yb)); n += 1
    return tot / max(n, 1)

def ode_predict_fn(model):
    def pf(X, t):
        model.eval()
        with torch.no_grad():
            Xb = torch.from_numpy(X.astype(np.float32)).to(DEVICE)
            tb = torch.from_numpy(t.astype(np.float32)).to(DEVICE)
            lg, _, _ = model(Xb, tb)
            return torch.sigmoid(lg).cpu().numpy()
    return pf

def run_ode(solver='dopri5', loss='focal', epochs=8, latent=8, hidden=32,
            max_train=15000, tag=None, decoupled=False):
    Xtr, ytr, ttr = get_arrays(df, feat_cols, split_info['train'])
    Xtr, ytr, ttr = subsample(Xtr, ytr, ttr, max_train)
    ttr = normalize_time(ttr)
    Xv, yv, tv = get_arrays(df, feat_cols, split_info['val'])
    tv = normalize_time(tv)
    model = LatentNeuralODE(
        input_dim=Xtr.shape[1], latent_dim=latent, hidden=hidden,
        solver=solver, use_ode=not decoupled,
    ).to(DEVICE)
    crit = FocalLoss() if loss == 'focal' else nn.BCEWithLogitsLoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    ds = TensorDataset(torch.from_numpy(Xtr), torch.from_numpy(ytr),
                       torch.from_numpy(ttr))
    loader = DataLoader(ds, batch_size=256, shuffle=True)
    label = ('Decoupled (no ODE)' if decoupled else f'Neural ODE ({solver}, {loss})')
    print(f'Training {label} on {Xtr.shape[0]} samples, {Xtr.shape[1]} features')
    best = float('inf')
    ckpt = f'E:\capstone/results/ode_model_{tag or label}.pt'
    for ep in range(1, epochs+1):
        tr = train_epoch(model, loader, opt, crit)
        vl = val_loss(model, Xv, yv, tv, crit)
        print(f'  epoch {ep:02d}: train={tr:.4f} val={vl:.4f}')
        if vl < best:
            best = vl; torch.save(model.state_dict(), ckpt)
    res = evaluate_model(ode_predict_fn(model), df, feat_cols, split_info)
    ov = res['overall']
    print(f'{label}: AUROC={ov["auroc"]:.4f} F1={ov["f1_macro"]:.4f} AUT={ov["aut"]:.4f}')
    rows = []
    for m in res['months']:
        pm = res['per_month'][m]
        rows.append({'model': label, 'month': m, 'auroc': pm['auroc'],
                     'f1_macro': pm['f1_macro'], 'support': pm['support'],
                     'malware': pm['malware'], 'benign': pm['benign']})
    rows.append({'model': label, 'month': 'OVERALL', 'auroc': ov['auroc'],
                 'f1_macro': ov['f1_macro'], 'aut': ov['aut']})
    path = f'E:\capstone/results/ode_metrics_{tag or label}.csv'
    pd.DataFrame(rows).to_csv(path, index=False)
    print('Saved', path, 'and', ckpt)
    return model

ode_dopri5 = run_ode(solver='dopri5', loss='focal', epochs=8, tag='dopri5_focal')
ode_rk4 = run_ode(solver='rk4', loss='focal', epochs=8, tag='rk4_focal')
ode_dec = run_ode(solver='dopri5', loss='focal', epochs=8, tag='decoupled', decoupled=True)

In [ ]:
def month_lbl(m):
    return 'Overall' if m == 'OVERALL' else pd.Period(m).strftime('%b %Y')

LABEL_MAP = {
    'XGBoost (static)': 'XGBoost (static)',
    'LightGBM (static)': 'LightGBM (static)',
    'MLP (static)': 'MLP (static)',
    'Neural ODE (dopri5, focal)': 'Neural ODE (dopri5)',
    'Neural ODE (rk4, focal)': 'Neural ODE (rk4)',
    'Decoupled (no ODE)': 'Decoupled (no ODE)',
}
PALETTE = {
    'XGBoost (static)': '#d62728', 'LightGBM (static)': '#ff7f0e',
    'MLP (static)': '#9467bd', 'Neural ODE (dopri5, focal)': '#2ca02c',
    'Neural ODE (rk4, focal)': '#1f77b4', 'Decoupled (no ODE)': '#7f7f7f',
}
STATIC = {'XGBoost (static)', 'LightGBM (static)', 'MLP (static)'}

frames = []
for p in sorted(glob.glob('E:\capstone/results/*metrics*.csv')):
    if 'comparison_table' in os.path.basename(p): continue
    frames.append(pd.read_csv(p))
metrics = pd.concat(frames, ignore_index=True)
test_months = sorted([m for m in metrics['month'].unique() if m != 'OVERALL'])

# ---- Plot 1: AUROC vs month ----
fig, ax = plt.subplots(figsize=(9, 5.5))
for name in metrics['model'].unique():
    sub = metrics[metrics['model'] == name].set_index('month').loc[test_months]
    if sub.empty: continue
    ax.plot(range(len(test_months)), sub['auroc'].values, marker='o', markersize=6,
            linewidth=2.2, label=LABEL_MAP.get(name, name), color=PALETTE.get(name),
            linestyle='--' if name in STATIC else '-', alpha=0.9 if name in STATIC else 1.0)
ax.set_xticks(range(len(test_months)))
ax.set_xticklabels([month_lbl(m) for m in test_months])
ax.set_ylim(0.4, 1.02); ax.set_ylabel('AUROC'); ax.set_xlabel('Test month')
ax.set_title('Month-by-Month AUROC (real metrics)')
ax.grid(True, alpha=0.3, linestyle=':'); ax.legend(fontsize=9, loc='lower left')
plt.tight_layout(); plt.savefig('E:\capstone/graphs/auroc_vs_month.png', dpi=150); plt.close()

# ---- Plot 2: AUT comparison ----
ov = metrics[metrics['month'] == 'OVERALL'].set_index('model')
models = [m for m in ov.index if m in LABEL_MAP]
aut_vals = [float(ov.loc[m,'aut']) if not np.isnan(ov.loc[m,'aut']) else 0.0 for m in models]
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
ax = axes[0]
bars = ax.bar(range(len(models)), aut_vals, color=[PALETTE.get(m,'#333') for m in models],
              width=0.6, edgecolor='k', linewidth=0.5, alpha=0.85)
ax.set_xticks(range(len(models)))
ax.set_xticklabels([LABEL_MAP[m] for m in models], rotation=20, ha='right', fontsize=9)
ax.set_ylabel('AUT'); ax.set_title('AUT per model (higher = more resilient)')
ax.grid(True, axis='y', alpha=0.3, linestyle=':')
for b, v in zip(bars, aut_vals):
    ax.text(b.get_x()+b.get_width()/2, v+0.02, f'{v:.2f}', ha='center', fontsize=8)
ax = axes[1]
for name in metrics['model'].unique():
    sub = metrics[(metrics['model']==name) & (metrics['month']!='OVERALL')].set_index('month').loc[test_months]
    if sub.empty: continue
    cum = np.cumsum(sub['auroc'].values.astype(float))
    ax.plot(range(len(test_months)), cum, marker='o', markersize=5, linewidth=2,
            label=LABEL_MAP.get(name, name), color=PALETTE.get(name),
            linestyle='--' if name in STATIC else '-')
ax.set_xticks(range(len(test_months)))
ax.set_xticklabels([month_lbl(m) for m in test_months], fontsize=9)
ax.set_ylabel('Cumulative AUROC'); ax.set_title('Cumulative AUROC over window')
ax.grid(True, alpha=0.3, linestyle=':'); ax.legend(fontsize=8, loc='upper left')
plt.tight_layout(); plt.savefig('E:\capstone/graphs/aut_comparison.png', dpi=150); plt.close()

# ---- Plot 3: UMAP latent migration (from saved dopri5 checkpoint) ----
ckpt = 'E:\capstone/results/ode_model_Neural ODE (dopri5, focal).pt'
if not os.path.exists(ckpt):
    ckpt = 'E:\capstone/results/ode_model_dopri5_focal.pt'
if os.path.exists(ckpt):
    n_samp = 900
    Xp, yp, tp = [], [], []
    rng = np.random.RandomState(RANDOM)
    for m in split_info['test_months']:
        idx = split_info['month_idx'][m]
        Xm = df.loc[idx, feat_cols].fillna(0).astype(np.float32).values
        ym = df.loc[idx, 'Label'].astype(np.float32).values
        tm = normalize_time(df.loc[idx, 'timestamp'].values)
        n = min(n_samp // len(split_info['test_months']), Xm.shape[0])
        sel = rng.choice(Xm.shape[0], size=n, replace=False)
        Xp.append(Xm[sel]); yp.append(ym[sel]); tp.append(tm[sel])
    Xs = np.vstack(Xp); ys = np.concatenate(yp); ts = np.concatenate(tp)
    mdel = LatentNeuralODE(input_dim=Xs.shape[1], latent_dim=8, hidden=32,
                           solver='dopri5', use_ode=True).to(DEVICE)
    mdel.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    mdel.eval()
    with torch.no_grad():
        _, z0, zt = mdel(torch.from_numpy(Xs).to(DEVICE),
                         torch.from_numpy(ts).to(DEVICE))
    z0, zt = z0.cpu().numpy(), zt.cpu().numpy()
    both = np.vstack([z0, zt])
    if HAS_UMAP:
        reducer = umap.UMAP(n_neighbors=15, min_dist=0.1, random_state=42)
        Z = reducer.fit_transform(both)
    else:
        Z = PCA(n_components=2, random_state=42).fit_transform(both)
    n = z0.shape[0]; Z0, Z1 = Z[:n], Z[n:]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    cols = {0: '#1f77b4', 1: '#d62728'}
    for ax, Zz, title in [(axes[0], Z0, 'Latent — initial z(t₀)'),
                          (axes[1], Z1, 'Latent — evolved z(t₁)')]:
        for cls in np.unique(ys):
            mask = ys == cls
            ax.scatter(Zz[mask,0], Zz[mask,1], c=cols[cls],
                       label='Benign' if cls==0 else 'Malware', s=22,
                       alpha=0.7, edgecolor='k', linewidth=0.25)
        ax.set_title(title); ax.set_xlabel('Component 1'); ax.set_ylabel('Component 2')
        ax.legend(fontsize=9); ax.grid(True, alpha=0.25, linestyle=':')
    plt.suptitle('Latent Space Migration: initial vs evolved (trained Neural ODE)', y=1.02)
    plt.tight_layout(); plt.savefig('E:\capstone/graphs/umap_latent_space.png', dpi=150); plt.close()
    print('UMAP plot saved')
else:
    print('Checkpoint not found; skipping UMAP')

print('All graphs saved to E:\capstone/graphs/')

In [ ]:
ov = metrics[metrics['month'] == 'OVERALL'].copy()
ov = ov.sort_values('auroc', ascending=False)
summary = pd.DataFrame({
    'model': [LABEL_MAP.get(r['model'], r['model']) for _, r in ov.iterrows()],
    'type': ['static' if r['model'] in STATIC else 'continuous' for _, r in ov.iterrows()],
    'AUROC': [round(float(r['auroc']), 4) for _, r in ov.iterrows()],
    'Macro F1': [round(float(r['f1_macro']), 4) for _, r in ov.iterrows()],
    'AUT': [round(float(r['aut']), 4) if not np.isnan(r['aut']) else None for _, r in ov.iterrows()],
})
summary.to_csv('E:\capstone/results/comparison_table.csv', index=False)
display(summary)

from IPython.display import Image, display as disp
for g in ['auroc_vs_month.png', 'aut_comparison.png', 'umap_latent_space.png']:
    path = f'E:\capstone/graphs/{g}'
    if os.path.exists(path):
        disp(Image(filename=path, width=800))
print('\nProof-of-work files in E:\capstone/results and E:\capstone/graphs')